In [1]:
import torch
import torch.nn as nn
from transformers import BertModel, BertTokenizer
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MAX_LEN = 256

In [3]:
class Bert_Classifier(nn.Module):
    def __init__(self, freeze_bert=False):
        super(Bert_Classifier, self).__init__()

        n_input = 768
        n_hidden = 50
        n_output = 5

        self.bert = BertModel.from_pretrained('bert-base-cased')

        self.classifier = nn.Sequential(
            nn.Linear(n_input, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_output)
        )
    
        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_emb = outputs.last_hidden_state[:, 0, :]
        logits  = self.classifier(cls_emb)
        return logits

In [4]:
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

model = Bert_Classifier()
model.load_state_dict(torch.load("cyberbullying_bert_model.pth", map_location=device))
model.to(device)
model.eval()

Some weights of the model checkpoint at bert-base-cased were not used when initializing BertModel: ['cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Bert_Classifier(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [5]:
def encode_text(texts):
    encoded = tokenizer.batch_encode_plus(
        texts,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors="pt"
    )
    return encoded['input_ids'], encoded['attention_mask']

In [7]:
def predict_text(model, text):
    input_ids, attention_mask = encode_text([text])
    input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        pred = torch.argmax(logits, dim=1).item()

    return pred

In [10]:
import pandas as pd

FILE_PATH = '../../../cyberbullying_tweets_transformers.csv'
df = pd.read_csv(FILE_PATH)
df = df.dropna(how='any')

mapping = df[['label', 'type']].drop_duplicates().sort_values('label')
print(mapping)

       label                 type
0          0               gender
7973       1             religion
15971      2  other_cyberbullying
23786      3                  age
31778      4            ethnicity


In [12]:
texts = [
    "Nigger",
    "Little girl",
    "Just tell you like girls",
    "Fuck you lesbian",
    "Fucking Muslim",
    "FUCK YOU!!"
]

for txt in texts:
    result = predict_text(model, txt)
    print(f"Text: {txt}")
    print(f"Predicted Category: {result}\n")

Text: Nigger
Predicted Category: 4

Text: Little girl
Predicted Category: 3

Text: Just tell you like girls
Predicted Category: 0

Text: Fuck you lesbian
Predicted Category: 0

Text: Fucking Muslim
Predicted Category: 1

Text: FUCK YOU!!
Predicted Category: 2

